In [2]:
import GEOparse

# tải & parse GSE10072 (có thể mất ~1 phút, nó tải 1 file về ./geo_data)
gse = GEOparse.get_GEO(geo="GSE10072", destdir="./geo_data", silent=True)

meta = gse.phenotype_data          # metadata: mỗi DÒNG = 1 mẫu (GSM)
print(meta.shape)                  # (số mẫu, số cột metadata)
print(meta.columns.tolist())       # tên các cột

(107, 44)
['title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'taxid_ch1', 'characteristics_ch1.0.Gender', 'characteristics_ch1.1.Age at Diagnosis', 'characteristics_ch1.2.Cigarette Smoking Status', 'characteristics_ch1.3.Stage:IIB', 'molecule_ch1', 'extract_protocol_ch1', 'label_ch1', 'label_protocol_ch1', 'hyb_protocol', 'scan_protocol', 'description', 'data_processing', 'platform_id', 'contact_name', 'contact_email', 'contact_phone', 'contact_laboratory', 'contact_department', 'contact_institute', 'contact_address', 'contact_city', 'contact_state', 'contact_zip/postal_code', 'contact_country', 'supplementary_file', 'relation', 'series_id', 'data_row_count', 'characteristics_ch1.3.Stage:IB', 'characteristics_ch1.3.Stage:IIA', 'characteristics_ch1.3.Stage:IIIA', 'characteristics_ch1.3.Stage:IIIB', 'characteristics_ch1.3.Stage:IV', 'characteristics_ch1.3.Stage:IA']


In [3]:
# xem 10 giá trị đầu của cột title — có nhúng mã bệnh nhân/số cặp không?
print(meta['title'].head(10).tolist())

# xem cột relation (nếu GEO ghi quan hệ mẫu ở đây)
print(meta['relation'].head(10).tolist())


['Lung Tumor_GT00006', 'Normal Lung_GT00006', 'Lung Tumor_GT00007', 'Normal Lung_GT00007', 'Lung Tumor_GT00022', 'Lung Tumor_GT00042', 'Lung Tumor_GT00059', 'Normal Lung_GT00059', 'Lung Tumor_GT00146', 'Normal Lung_GT00146']
['Reanalyzed by: GSE60486', 'Reanalyzed by: GSE60486', 'Reanalyzed by: GSE60486', 'Reanalyzed by: GSE60486', 'Reanalyzed by: GSE60486', 'Reanalyzed by: GSE60486', 'Reanalyzed by: GSE60486', 'Reanalyzed by: GSE60486', 'Reanalyzed by: GSE60486', 'Reanalyzed by: GSE60486']


In [4]:
# tách title tại dấu "_" cuối: trước là loại mô, sau là mã bệnh nhân
meta['patient'] = meta['title'].str.split('_').str[-1]      # lấy phần SAU cùng → GT00006
meta['tissue']  = meta['title'].str.split('_').str[0]       # lấy phần ĐẦU   → "Lung Tumor" / "Normal Lung"

print(meta[['title', 'patient', 'tissue']].head(6))

                         title  patient       tissue
GSM254625   Lung Tumor_GT00006  GT00006   Lung Tumor
GSM254626  Normal Lung_GT00006  GT00006  Normal Lung
GSM254627   Lung Tumor_GT00007  GT00007   Lung Tumor
GSM254628  Normal Lung_GT00007  GT00007  Normal Lung
GSM254629   Lung Tumor_GT00022  GT00022   Lung Tumor
GSM254630   Lung Tumor_GT00042  GT00042   Lung Tumor


In [5]:
# với mỗi bệnh nhân, tập hợp các loại mô họ có
per_patient = meta.groupby('patient')['tissue'].apply(set)
print("Tổng số bệnh nhân:", per_patient.shape[0])
print(per_patient.head(8))


Tổng số bệnh nhân: 74
patient
GT00006    {Lung Tumor, Normal Lung}
GT00007    {Lung Tumor, Normal Lung}
GT00022                 {Lung Tumor}
GT00042                 {Lung Tumor}
GT00059    {Lung Tumor, Normal Lung}
GT00146    {Lung Tumor, Normal Lung}
GT01001    {Lung Tumor, Normal Lung}
GT01003    {Lung Tumor, Normal Lung}
Name: tissue, dtype: object


In [6]:
# đếm theo số loại mô mỗi bệnh nhân có
n_paired    = (per_patient.apply(len) == 2).sum()          # có đủ 2 loại → cặp
n_tumor_only  = (per_patient == {'Lung Tumor'}).sum()      # chỉ u
n_normal_only = (per_patient == {'Normal Lung'}).sum()     # chỉ lành

print("Cặp đủ (có cả u & lành):", n_paired)
print("Chỉ u (lẻ):", n_tumor_only)
print("Chỉ lành (lẻ):", n_normal_only)
print("Tổng kiểm:", n_paired + n_tumor_only + n_normal_only, "(phải = 74)")

Cặp đủ (có cả u & lành): 33
Chỉ u (lẻ): 25
Chỉ lành (lẻ): 16
Tổng kiểm: 74 (phải = 74)


In [7]:
# ma trận biểu hiện: dòng = probe, cột = mẫu
expr = gse.pivot_samples('VALUE')
print("Kích thước:", expr.shape)          # (số probe, 107 mẫu)
print("Min:", expr.min().min())           # giá trị nhỏ nhất toàn ma trận
print("Max:", expr.max().max())           # giá trị lớn nhất
print("Median:", expr.stack().median())   # trung vị

Kích thước: (22283, 107)
Min: 3.670126391
Max: 15.24822482
Median: 7.426137187


In [8]:
import os

# biến Giai đoạn 0 còn sống? (NameError → kernel bị reset, chạy lại cells Giai đoạn 0 trước)
print("expr:", expr.shape)                                   # kỳ vọng (22283, 107)
print("meta có patient/tissue:", {'patient','tissue'} <= set(meta.columns))  # kỳ vọng True

base = os.path.expanduser("~/python_practice/luad_project/results")
expr.to_csv(f"{base}/expression_matrix_log2.csv")
print("Đã lưu ma trận — MB:", round(os.path.getsize(f'{base}/expression_matrix_log2.csv')/1e6, 1))


expr: (22283, 107)
meta có patient/tissue: True
Đã lưu ma trận — MB: 28.6


In [9]:
info = meta[['patient', 'tissue']]      # đúng 2 cột mô hình cần
info.to_csv(f"{base}/sample_info.csv")  # index = GSM → thành cột đầu tiên
print(info.shape)                        # kỳ vọng (107, 2)
print(info.head(3))


(107, 2)
           patient       tissue
GSM254625  GT00006   Lung Tumor
GSM254626  GT00006  Normal Lung
GSM254627  GT00007   Lung Tumor
